# LangChain RAG Agent (OpenRouter 免费版)

This notebook demonstrates how to build a traditional RAG (Retrieval-Augmented Generation) Agent using LangChain with **OpenRouter's free model** (`openai/gpt-oss-120b:free`).

**主要改动：**
- LLM: DashScope → OpenRouter (免费)
- 模型: `qwen-plus` → `openai/gpt-oss-120b:free`
- Embeddings: 继续使用 DashScope (或可改用其他免费方案)

## 1. Import Required Libraries

In [9]:
import os
import re
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import DashScopeEmbeddings
from langchain_milvus import Milvus
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, MessagesState, START, END
# PDF loader
from langchain_community.document_loaders import PyPDFLoader, TextLoader
# DashScope SDK - 用于 Embeddings
import dashscope

## 2. Load Environment Variables and Configuration

使用阿里云 DashScope 的 LLM 和 Embeddings

In [ ]:
# ========== DashScope API Configuration (用于 LLM 和 Embeddings) ==========
# ⚠️ Important: Replace "YOUR_DASHSCOPE_API_KEY" below with your actual API key
# Example: key = "sk-1234567890abcdef..."
# PRIORITY: Code value > Environment variable
key = ""

# Only load .env if key is not set in code
if not key or key == "YOUR_DASHSCOPE_API_KEY":
    load_dotenv()  # Load .env file only if needed
    key = os.getenv("DASHSCOPE_API_KEY")
    if not key:
        raise ValueError(
            "❌ API key not set!\n"
            "Please replace 'YOUR_DASHSCOPE_API_KEY' in the code with your actual API key\n"
            "Or: Set DASHSCOPE_API_KEY=your_api_key in .env file"
        )
else:
    # Key is set in code, but still load .env for other variables (Milvus, etc.)
    load_dotenv()

# Clean API key (remove possible spaces and quotes)
if key:
    key = key.strip().strip('"').strip("'")

# base_url should be the API address, not the API key!
base_url = os.getenv("DASHSCOPE_API_BASE") or "https://dashscope.aliyuncs.com/compatible-mode/v1"

# Display API key information (for debugging)
print("=" * 60)
print("🔍 API Key Check")
print("=" * 60)
if key and key != "YOUR_DASHSCOPE_API_KEY":
    print(f"✅ API key is set")
    print(f"   Length: {len(key)} characters")
    print(f"   First 10 characters: {key[:10]}...")
    print(f"   Last 4 characters: ...{key[-4:]}")
    print(f"   Starts with 'sk-': {key.startswith('sk-')}")
    if not key.startswith('sk-'):
        print("   ⚠️ Warning: API key should usually start with 'sk-'")
    print(f"   base_url: {base_url}")
    # Check if .env file exists and might have conflicting key
    env_key = os.getenv("DASHSCOPE_API_KEY")
    if env_key and env_key != key:
        print(f"   ⚠️ Note: .env file has different key (ending with ...{env_key[-4:]})")
        print(f"   → Using code value (ending with ...{key[-4:]})")
else:
    print("❌ API key not set correctly!")
    print("   Please replace 'YOUR_DASHSCOPE_API_KEY' in the code with your actual API key")
print("=" * 60)

# ========== Milvus Connection Configuration ==========
# Method 1: Enter directly in code (recommended for testing)
milvus_uri = "https://in03-13d3fc765a723cc.serverless.gcp-us-west1.cloud.zilliz.com"
milvus_user = "db_13d3fc765a723cc"
milvus_password = "Ew7|K2USgunQOqnb"

# Method 2: Read from environment variables (if not set above, will try to read from .env file)
if not milvus_uri or milvus_uri == "YOUR_MILVUS_URI" or milvus_uri == "https://xxxxxxxx.serverless.gcp-us-west1.cloud.zilliz.com":
    milvus_uri = os.getenv("MILVUS_URI")
    milvus_user = os.getenv("MILVUS_USER")
    milvus_password = os.getenv("MILVUS_PASSWORD")

🔍 API Key Check
✅ OpenRouter API key is set
   Length: 73 characters
   First 10 characters: sk-or-v1-6...
   Last 4 characters: ...3156
   Model: qwen/qwen3-next-80b-a3b-instruct:free
   Base URL: https://openrouter.ai/api/v1
✅ DashScope API key is set
   Length: 35 characters
   First 10 characters: sk-3fc5861...
   Last 4 characters: ...610d
   Starts with 'sk-': True
   base_url: https://dashscope.aliyuncs.com/compatible-mode/v1


## 3. Initialize LLM (DashScope)

In [15]:
# Initialize LLM with OpenRouter
# 使用 OpenRouter 的免费模型：openai/gpt-oss-120b:free
# 参考：https://openrouter.ai/openai/gpt-oss-120b:free

graph_llm = ChatOpenAI(
    temperature=0,
    model_name=openrouter_model,
    api_key=openrouter_api_key,
    base_url=openrouter_base_url
)

llm = ChatOpenAI(
    temperature=0,
    model_name=openrouter_model,
    api_key=openrouter_api_key,
    base_url=openrouter_base_url
)

print("✅ LLM 初始化成功")
print(f"   模型: {openrouter_model}")
print(f"   提供商: OpenRouter (免费)")

✅ LLM 初始化成功
   模型: openai/gpt-oss-120b:free
   提供商: OpenRouter (免费)


## 4. Load Documents

Supports TXT and PDF formats

In [16]:
# Step 1: Load documents
# Supports TXT and PDF formats
file_path = 'Cytokine Regulation and Function in T Cells.pdf'  # Can be changed to '../doc/company.pdf' to load PDF

# Select loader based on file extension
if file_path.endswith('.pdf'):
    loader = PyPDFLoader(file_path)
    documents = loader.load()
    print(f"PDF document loaded, {len(documents)} pages")
    total_chars = sum(len(doc.page_content) for doc in documents)
    print(f"Total content length: {total_chars} characters")
else:
    # Load TXT file
    loader = TextLoader(file_path, encoding='utf-8')
    documents = loader.load()
    print(f"TXT document loaded, content length: {len(documents[0].page_content)} characters")

PDF document loaded, 26 pages
Total content length: 93803 characters


## 5. Text Splitting

In [17]:
# Step 2: Text splitting and cleaning
def clean_text(text):
    """Clean text, remove characters that may cause encoding issues"""
    if not text:
        return ""
    # Remove control characters (except newline, tab, and carriage return)
    text = re.sub(r'[\x00-\x08\x0b-\x0c\x0e-\x1f\x7f-\x9f]', '', text)
    # Remove zero-width characters
    text = re.sub(r'[\u200b-\u200f\u202a-\u202e\u2060-\u206f]', '', text)
    # Ensure text can be properly encoded as UTF-8
    try:
        text.encode('utf-8')
    except UnicodeEncodeError:
        # If there are still issues, use error handling
        text = text.encode('utf-8', errors='ignore').decode('utf-8')
    return text

# Clean original documents first
for doc in documents:
    doc.page_content = clean_text(doc.page_content)
    # Clean metadata
    if doc.metadata:
        cleaned_metadata = {}
        for key, value in doc.metadata.items():
            if isinstance(value, str):
                cleaned_metadata[key] = clean_text(value)
            else:
                cleaned_metadata[key] = value
        doc.metadata = cleaned_metadata

chunk_size = 250
chunk_overlap = 30
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size, 
    chunk_overlap=chunk_overlap
)

splits = text_splitter.split_documents(documents)
print(f"Document split into {len(splits)} chunks")

# Clean split documents again and filter empty documents
cleaned_splits = []
for doc in splits:
    cleaned_content = clean_text(doc.page_content)
    if cleaned_content.strip():  # Only keep non-empty documents
        doc.page_content = cleaned_content
        cleaned_splits.append(doc)

splits = cleaned_splits
print(f"Remaining {len(splits)} valid chunks after cleaning")

Document split into 485 chunks
Remaining 485 valid chunks after cleaning


## 6. Initialize Embeddings (继续使用 DashScope)

In [18]:
# Step 3: Initialize Embeddings
# 
# 💡 免费 Embeddings API 选项：
# 1. HuggingFace Inference API (免费，但有限制)
#    from langchain_community.embeddings import HuggingFaceInferenceAPIEmbeddings
#    embeddings = HuggingFaceInferenceAPIEmbeddings(
#        api_key="your_hf_token",
#        model_name="sentence-transformers/all-MiniLM-L6-v2"
#    )
#
# 2. Jina AI (免费额度: 100万字符/月)
#    from langchain_community.embeddings import JinaEmbeddings
#    embeddings = JinaEmbeddings(jina_api_key="your_jina_key")
#
# 3. 继续使用 DashScope (需要 API Key)

# 当前使用 DashScope Embeddings
if not dashscope_key:
    print("=" * 60)
    print("⚠️  DashScope API Key 未设置")
    print("=" * 60)
    print("💡 提示：可以使用免费的 Embeddings 替代方案")
    print("   1. HuggingFace Inference API (免费)")
    print("   2. Jina AI (免费额度: 100万字符/月)")
    print("=" * 60)
    raise ValueError("❌ DashScope API Key 未设置！请先设置 DASHSCOPE_API_KEY")

# 设置 dashscope.api_key
dashscope.api_key = dashscope_key

print("=" * 60)
print("🔧 DashScope Embeddings Configuration")
print("=" * 60)
print(f"✅ dashscope.api_key 已设置")

# 初始化 Embeddings
embeddings = DashScopeEmbeddings(
    model="text-embedding-v4",
    dashscope_api_key=dashscope_key
)

# 测试 Embeddings
try:
    test_result = embeddings.embed_query("test")
    print("✅ DashScope Embeddings 初始化成功")
    print(f"   测试结果维度: {len(test_result)}")
except Exception as e:
    print(f"❌ Embeddings 测试失败: {e}")
    raise

print("=" * 60)

🔧 DashScope Embeddings Configuration
✅ dashscope.api_key 已设置
✅ DashScope Embeddings 初始化成功
   测试结果维度: 1024


## 7. Build Vector Index and Store to Milvus

In [19]:
# Step 4: Build vector index and store to Milvus
# ⚠️ 重要：如果 collection 已存在，可以直接加载，不需要重新向量化！

COLLECTION_NAME = "company_milvus"  # 你的 collection 名称

# 首先测试 Milvus 连接
print("=" * 60)
print("🔌 测试 Milvus 连接...")
print("=" * 60)

# Milvus 连接配置（使用 URI + User + Password）
if not milvus_uri or milvus_uri == "https://xxxxxxxx.serverless.gcp-us-west1.cloud.zilliz.com" or not milvus_user or not milvus_password:
    print("❌ Milvus 连接信息未配置！")
    print("\n💡 请在代码中填写：")
    print("   milvus_uri = 'your_uri'")
    print("   milvus_user = 'your_user'")
    print("   milvus_password = 'your_password'")
    print("=" * 60)
    raise ValueError("Milvus 连接信息未配置")

# 构建连接参数
connection_args = {
    "uri": milvus_uri,
    "user": milvus_user,
    "password": milvus_password,
}

print("=" * 60)
print("📝 Milvus 连接配置")
print("=" * 60)
print(f"   URI: {milvus_uri}")
print(f"   User: {milvus_user}")
print(f"   Password: {'*' * len(milvus_password)}")
print("=" * 60)

# 首先使用 MilvusClient 检查 collection 是否存在（参考 Zilliz 文档）
print("=" * 60)
print("📂 检查 Milvus collection 是否存在...")
print("=" * 60)

try:
    from pymilvus import MilvusClient
    
    # 创建 MilvusClient 连接（用于检查 collection）
    client = MilvusClient(
        uri=connection_args["uri"],
        user=connection_args["user"],
        password=connection_args["password"],
    )
    
    # 列出所有 collection（参考 Zilliz 文档的方法）
    collections = client.list_collections()
    print(f"   当前已有的 collections: {collections}")
    
    if COLLECTION_NAME in collections:
        print(f"✅ Collection '{COLLECTION_NAME}' 已存在！")
        
        # 获取 collection 的详细信息
        try:
            collection_info = client.describe_collection(collection_name=COLLECTION_NAME)
            print(f"   Collection 信息: {collection_info}")
        except Exception as e:
            print(f"   ⚠️  无法获取详细信息: {e}")
        
        print("=" * 60)
        client.close()  # 先关闭检查用的 client
        
        # 方法1: 加载已有的 collection（不需要重新向量化）
        print("📂 加载已有的 collection...")
        # 确保 Milvus 类已导入（从 langchain_milvus）
        from langchain_milvus import Milvus
        
        vectorstore = Milvus(
            embedding_function=embeddings,
            collection_name=COLLECTION_NAME,
            connection_args=connection_args
        )
        
        # 测试是否能正常检索
        test_retriever = vectorstore.as_retriever(search_kwargs={"k": 1})
        test_docs = test_retriever.invoke("test")
        
        print(f"✅ 成功加载已有的 collection: {COLLECTION_NAME}")
        print(f"   ⚡ 无需重新向量化，直接使用已有数据！")
        print("=" * 60)
        
    else:
        print(f"⚠️  Collection '{COLLECTION_NAME}' 不存在")
        print("   将创建新的 collection 并向量化文档...")
        print("=" * 60)
        client.close()  # 关闭检查用的 client
        
        # 方法2: 创建新的 collection（首次运行）
        print("🔨 开始创建新的 collection 并向量化文档...")
        print("   （这可能需要几分钟，取决于文档数量）")
        
        # 确保 Milvus 类已导入
        from langchain_milvus import Milvus
        
        vectorstore = Milvus.from_documents(
            documents=splits,
            collection_name=COLLECTION_NAME,
            embedding=embeddings,
            connection_args=connection_args
        )
        
        print(f"✅ 创建新的 collection: {COLLECTION_NAME}")
        print(f"   已向量化并存储 {len(splits)} 个文档块")
        print("=" * 60)
    
except Exception as e:
    print(f"❌ 检查 collection 时出错: {type(e).__name__}: {str(e)[:200]}")
    print("=" * 60)
    print("💡 可能的原因：")
    print("1. Milvus 连接失败（检查 URI、用户名、密码）")
    print("2. 网络连接问题")
    print("3. Milvus 服务不可用")
    print("=" * 60)
    raise

🔌 测试 Milvus 连接...
📝 Milvus 连接配置
   URI: https://in03-13d3fc765a723cc.serverless.gcp-us-west1.cloud.zilliz.com
   User: db_13d3fc765a723cc
   Password: ****************
📂 检查 Milvus collection 是否存在...
   当前已有的 collections: ['company_milvus']
✅ Collection 'company_milvus' 已存在！
   Collection 信息: {'collection_name': 'company_milvus', 'auto_id': True, 'num_shards': 1, 'description': '', 'fields': [{'field_id': 100, 'name': 'text', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 65535}}, {'field_id': 101, 'name': 'pk', 'description': '', 'type': <DataType.INT64: 5>, 'params': {}, 'auto_id': True, 'is_primary': True}, {'field_id': 102, 'name': 'vector', 'description': '', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 1024}}, {'field_id': 103, 'name': 'producer', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 65535}}, {'field_id': 104, 'name': 'creator', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length':

## 8. Create RAG Chain

In [20]:
# Step 5: Create RAG Chain
prompt = PromptTemplate(
    template="""You are an immunology experiment-planning assistant.
Design an executable experimental plan using ONLY the provided context. Do NOT invent parameters (e.g., concentrations, incubation times, catalog numbers, instrument models) unless explicitly stated in the context.

Rules:
1) If the context has relevant info, propose a minimal, actionable plan tailored to the goal.
2) If critical details are missing, ask up to 3 clarifying questions (only the most critical).
3) Keep the response concise, but prioritize actionability over being short.

Question: {question}
Context: {context}

Answer in this format:
- Goal:
- Hypothesis:
- Minimal plan (3-7 steps):
- Controls:
- Readouts:
- Missing critical info (if any):
- Clarifying questions (0-3):""",
    input_variables=["question", "context"],
)

rag_chain = prompt | graph_llm | StrOutputParser()
print("RAG Chain 已创建（使用 OpenRouter 免费模型）")

RAG Chain 已创建（使用 OpenRouter 免费模型）


## 9. Test RAG Chain

In [ ]:
# Step 6: Test RAG Chain
question = "What CD4+ T helper subsets are discussed in this article?"

# 1) Retrieve more chunks for better coverage
retriever = vectorstore.as_retriever(search_kwargs={"k": 8})
docs = retriever.invoke(question)

# 2) Basic debug: show what was retrieved
print("=" * 70)
print(f"Retrieved {len(docs)} document chunks")
print("=" * 70)

if not docs:
    print("No documents retrieved.")
else:
    for i, doc in enumerate(docs[:5], 1):
        preview = doc.page_content.replace("\n", " ")
        preview = (preview[:300] + "...") if len(preview) > 300 else preview
        meta = getattr(doc, "metadata", {}) or {}
        print(f"\nChunk {i} | len={len(doc.page_content)} | meta={meta}")
        print(f"Preview: {preview}")

print("=" * 70)

# 3) Deduplicate + build context
seen = set()
unique_texts = []
for doc in docs:
    text = (doc.page_content or "").strip()
    if not text:
        continue
    key = text[:200]
    if key in seen:
        continue
    seen.add(key)
    unique_texts.append(text)

# 4) Truncate total context
MAX_CONTEXT_CHARS = 6000
context = "\n\n".join(unique_texts)
context = context[:MAX_CONTEXT_CHARS]

# 5) Run RAG Chain (使用 DashScope LLM)
print("\n🤖 正在使用 DashScope LLM 生成答案...")

generation = rag_chain.invoke({"context": context, "question": question})

print("\n✅ Generated answer:")
print(generation)

Retrieved 8 document chunks

Chunk 1 | len=190 | meta={'moddate': '2026-02-03T20:39:35+00:00', 'producer': 'Acrobat Distiller 10.1.10 (Windows); modified using OpenPDF 2.0.4', 'page_label': '64', 'creationdate': '2021-04-13T08:34:34+05:30', 'title': '', 'subject': '', 'author': '', 'total_pages': 26, 'page': 13, 'source': 'Cytokine Regulation and Function in T Cells.pdf', 'keywords': '', 'creator': 'LaTeX with hyperref package', 'pk': 464003139613949202}
Preview: In addition to the four well-established subsets of CD4+ effector T cells, i.e., Th1, Th2, Th17, and Tfh cells,other populations of cytokine-producing CD4+ T cells have been described in the

Chunk 2 | len=190 | meta={'moddate': '2026-02-03T20:39:35+00:00', 'producer': 'Acrobat Distiller 10.1.10 (Windows); modified using OpenPDF 2.0.4', 'page_label': '64', 'creationdate': '2021-04-13T08:34:34+05:30', 'title': '', 'subject': '', 'author': '', 'total_pages': 26, 'page': 13, 'source': 'Cytokine Regulation and Function in T Cells.

RateLimitError: Error code: 429 - {'error': {'message': 'Provider returned error', 'code': 429, 'metadata': {'raw': 'openai/gpt-oss-120b:free is temporarily rate-limited upstream. Please retry shortly, or add your own key to accumulate your rate limits: https://openrouter.ai/settings/integrations', 'provider_name': 'OpenInference', 'is_byok': False}}, 'user_id': 'user_39DranKki6CPQVlyCy7IWRCjudN'}

## 10. Define AgentState

In [ ]:
# Step 7: Define AgentState
class AgentState(MessagesState):
    next: str

## 11. Create Traditional RAG Agent Node

In [ ]:
# Step 8: Create traditional RAG Agent node
def vec_kg(state: AgentState):
    last_msg = state["messages"][-1]
    question = last_msg.content

    prompt = PromptTemplate(
        template="""You are an immunology experiment-planning assistant.
Design an executable experimental plan using ONLY the retrieved context. Do NOT invent parameters (e.g., concentrations, incubation times, catalog numbers, instrument models) unless explicitly stated in the context.

Rules:
1) If the context has relevant info, propose a minimal, actionable plan tailored to the goal.
2) If critical details are missing, ask up to 3 clarifying questions (only the most critical).
3) Keep the response concise, but prioritize actionability over being short.

Question: {question}
Context: {context}

Answer in this format:
- Goal:
- Hypothesis:
- Minimal plan (3-7 steps):
- Controls:
- Readouts:
- Missing critical info (if any):
- Clarifying questions (0-3):""",
        input_variables=["question", "context"],
    )

    rag_chain = prompt | graph_llm | StrOutputParser()

    retriever = vectorstore.as_retriever(search_kwargs={"k": 8})
    docs = retriever.invoke(question)

    seen = set()
    unique_texts = []
    for d in docs:
        text = (d.page_content or "").strip()
        if not text:
            continue
        sig = text[:200]
        if sig in seen:
            continue
        seen.add(sig)
        unique_texts.append(text)

    context = "\n\n".join(unique_texts)
    MAX_CONTEXT_CHARS = 6000
    context = context[:MAX_CONTEXT_CHARS]

    generation = rag_chain.invoke({"context": context, "question": question})

    final_response = [HumanMessage(content=generation, name="vec_kg")]
    return {"messages": final_response}

## 12. Test vec_kg Node

In [ ]:
# Step 9: Test vec_kg node
test_state = AgentState(
    messages=[
        HumanMessage(
            content="Based on this article, design a minimal experiment to study CD4+ T helper cell differentiation."
        )
    ]
)

result = vec_kg(test_state)

print("RAG Agent response (使用 OpenRouter 免费模型):")
for msg in result["messages"]:
    print("-" * 60)
    print(msg.content)